# Week 6 — Query transformation and re-ranking

Extends the Week 5 corpus and 10 frozen queries. Compares baseline MiniLM retrieval with Gemini Flash-Lite multi-query retrieval + reciprocal-rank fusion + local cross-encoder re-ranking. Reports P@3, R@3, Hit@1, MRR, nDCG@3, per-query deltas, and a regression case.


In [ ]:
import sys, subprocess, importlib.util
mods={"numpy":"numpy","pandas":"pandas","sentence_transformers":"sentence-transformers","faiss":"faiss-cpu","google.genai":"google-genai","dotenv":"python-dotenv"}
need=[p for m,p in mods.items() if importlib.util.find_spec(m) is None]
if need: subprocess.check_call([sys.executable,"-m","pip","install",*need])


In [ ]:
import os,json,time,math
from pathlib import Path
from collections import defaultdict
import numpy as np,pandas as pd,faiss
from sentence_transformers import SentenceTransformer,CrossEncoder
from IPython.display import display,Markdown
TOP_K=3; CANDIDATE_K=8
EMBED_MODEL="sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL="cross-encoder/ms-marco-MiniLM-L-6-v2"
GEMINI_MODEL="gemini-3.5-flash-lite"
CACHE_PATH=Path("week6_query_cache.json")


In [ ]:
DOCUMENTS=[
("D01","Request intake",[("S1","Required fields","A new operations request must include a short title, requester name, business team, problem statement, and desired outcome. The requester can attach example files but attachments are optional."),("S2","Duplicate requests","Before opening a new request, search the request list by title and team. If an existing request covers the same problem, add a comment to that request rather than creating a duplicate.")]),
("D02","Triage and priority",[("S1","Priority definitions","Priority 1 means a production service is unavailable to multiple people. Priority 2 means a blocked business process with a manual workaround. Priority 3 means an improvement or a question without an active outage."),("S2","First review","The operations coordinator checks completeness and assigns an owner within two business days. Missing intake fields are sent back to the requester before priority is assigned.")]),
("D03","Approval workflow",[("S1","Approval conditions","A request that changes customer-facing content or grants access to protected data requires manager approval before implementation. An internal wording correction with no data access change does not require approval."),("S2","Rejected approval","When a manager rejects a request, record the reason and set approval result to Rejected. The owner may revise the request and seek a new approval; rejection does not mean it is complete.")]),
("D04","Delivery and review",[("S1","Human review","Generated drafts must be reviewed for factual correctness by the assigned owner. A workflow must not automatically send a draft to customers or publish it internally before a person approves the final text."),("S2","Completion","Set status to Completed only after the requester accepts the delivered result and the owner records the outcome. Saving a generated draft alone does not close a request.")]),
("D05","SharePoint file handling",[("S1","Source files","Store source files in the team SharePoint document library. In the request, link the specific file and record its version; do not paste the full file into the request description."),("S2","Access permissions","Requesters may see their own request and shared team files. An owner checks a recipient's access before sharing a source link; a link does not grant permission by itself.")]),
("D06","Automation monitoring",[("S1","Flow failure","If a Power Automate run fails, inspect its run history and note the failed action and error in the request. Retry only after correcting the cause, to avoid sending duplicate notifications."),("S2","Success checks","For a new automation, test one ordinary case and one missing-field case. Record expected versus observed results and ask the owner to sign off before activation.")]),
("D07","Data safeguards",[("S1","Sensitive content","Remove customer identifiers and confidential attachments from example prompts unless their use is authorized. Use a small, sanitized excerpt when a full document is unnecessary."),("S2","Retention","Keep the request's decision and final outcome in the request list. Delete temporary test exports when validation finishes according to the team's retention schedule.")]),
("D08","Notifications and escalation",[("S1","Status updates","The owner posts a status update on the request when work starts and when the expected delivery date changes. Routine updates are visible to the requester in the list."),("S2","Escalation","Escalate a production outage to the on-call owner immediately and record the escalation time. A non-outage item waiting more than two business days for assignment goes to the operations coordinator.")])];
CHUNKS=[{"chunk_id":f"{d}-{s}","title":t,"heading":h,"body":b,"text":f"{t} — {h}. {b}"} for d,t,secs in DOCUMENTS for s,h,b in secs]; by_id={c["chunk_id"]:c for c in CHUNKS}
TESTS=[("Q01","What fields must a requester provide when opening a request?",{"D01-S1"}),("Q02","What should I do if someone has already reported the same problem?",{"D01-S2"}),("Q03","Which priority applies when a process is blocked but a workaround exists?",{"D02-S1"}),("Q04","How soon is an owner assigned, and what happens to an incomplete request?",{"D02-S2"}),("Q05","When does a request require manager approval?",{"D03-S1"}),("Q06","What happens after a manager rejects a request?",{"D03-S2"}),("Q07","Can an AI draft be published or emailed without a person checking it?",{"D04-S1"}),("Q08","When can the owner mark the request Completed?",{"D04-S2"}),("Q09","How do I investigate a failed Power Automate flow before retrying?",{"D06-S1"}),("Q10","How quickly should a production outage be escalated?",{"D08-S2"})]; display(pd.DataFrame([{"qid":q,"query":x,"gold":next(iter(g))} for q,x,g in TESTS]))


In [ ]:
embedder=SentenceTransformer(EMBED_MODEL); texts=[c["text"] for c in CHUNKS]; V=embedder.encode(texts,normalize_embeddings=True,show_progress_bar=False).astype("float32"); index=faiss.IndexFlatIP(V.shape[1]); index.add(V)
def retrieve(q,k=TOP_K):
 v=embedder.encode([q],normalize_embeddings=True,show_progress_bar=False).astype("float32"); s,i=index.search(v,k); return [{"chunk_id":CHUNKS[j]["chunk_id"],"score":float(sc),"rank":r+1} for r,(j,sc) in enumerate(zip(i[0],s[0]))]
baseline={qid:retrieve(q) for qid,q,_ in TESTS}; display(pd.DataFrame([{"qid":qid,"baseline":[x["chunk_id"] for x in baseline[qid]]} for qid,_,_ in TESTS]))


## Gemini multi-query transformation

Two rewrites per query are cached in `week6_query_cache.json`; an empty cache uses at most 20 Gemini calls.

API-key lookup order:
1. Google Colab Secrets (`GEMINI_API_KEY`)
2. Environment variable (`GEMINI_API_KEY`)
3. Local JupyterLab `.env` file

For compatibility, the common misspelling `GEMENI_API_KEY` is also recognized, but `GEMINI_API_KEY` is preferred.


In [ ]:
from dotenv import load_dotenv, find_dotenv

def load_gemini_key():
    # 1. Colab Secrets first.
    try:
        from google.colab import userdata
        for name in ("GEMINI_API_KEY", "GEMENI_API_KEY"):
            try:
                value = userdata.get(name)
                if value:
                    return value.strip(), f"Colab Secrets ({name})"
            except Exception:
                pass
    except ImportError:
        pass

    # 2. Existing environment variable.
    for name in ("GEMINI_API_KEY", "GEMENI_API_KEY"):
        value = os.getenv(name)
        if value:
            return value.strip(), f"environment variable ({name})"

    # 3. Local JupyterLab .env, searching from the current working directory upward.
    env_file = find_dotenv(usecwd=True)
    if env_file:
        load_dotenv(env_file, override=False)
        for name in ("GEMINI_API_KEY", "GEMENI_API_KEY"):
            value = os.getenv(name)
            if value:
                return value.strip(), f".env ({Path(env_file).name}, {name})"

    return None, None

cache=json.loads(CACHE_PATH.read_text()) if CACHE_PATH.exists() else {}
key,key_source=load_gemini_key()
needs_live_calls=any(len(cache.get(qid,[]))<2 for qid,_,_ in TESTS)

if key_source:
    print(f"Gemini API key loaded from {key_source}.")
elif needs_live_calls:
    raise RuntimeError(
        "Gemini API key not found. Add GEMINI_API_KEY to Colab Secrets "
        "(and enable notebook access), or put GEMINI_API_KEY in a local .env file for JupyterLab."
    )
else:
    print("Complete query cache found; Gemini API key is not required for this run.")

if key:
 from google import genai; client=genai.Client(api_key=key)
 def rewrite(q,n):
  p=f"Rewrite this semantic-search query using likely operations-policy vocabulary while preserving intent exactly. Return only one concise query. Variant {n}: {q}"; return client.models.generate_content(model=GEMINI_MODEL,contents=p).text.strip().strip(chr(34))
 for qid,q,_ in TESTS:
  a=cache.get(qid,[])
  while len(a)<2: a.append(rewrite(q,len(a)+1)); cache[qid]=a; CACHE_PATH.write_text(json.dumps(cache,indent=2)); time.sleep(.25)
transformed={qid:cache[qid][:2] for qid,_,_ in TESTS}; display(pd.DataFrame([{"qid":qid,"original":q,"rewrite_1":transformed[qid][0],"rewrite_2":transformed[qid][1]} for qid,q,_ in TESTS]))


In [ ]:
def rrf(lists,k=60):
 d=defaultdict(float)
 for L in lists:
  for x in L:d[x["chunk_id"]]+=1/(k+x["rank"])
 return [cid for cid,_ in sorted(d.items(),key=lambda z:z[1],reverse=True)]
reranker=CrossEncoder(RERANK_MODEL)
def final(q,rw):
 cand=rrf([retrieve(q,CANDIDATE_K)]+[retrieve(x,CANDIDATE_K) for x in rw])[:CANDIDATE_K]; scores=reranker.predict([(q,by_id[c]["text"]) for c in cand]); return [c for c,_ in sorted(zip(cand,scores),key=lambda z:float(z[1]),reverse=True)[:TOP_K]]
finals={qid:final(q,transformed[qid]) for qid,q,_ in TESTS}
def m(ids,g):
 hits=[int(x in g) for x in ids[:3]]; rr=next((1/i for i,x in enumerate(ids[:3],1) if x in g),0); dcg=sum(rel/math.log2(i+2) for i,rel in enumerate(hits)); return {"p3":sum(hits)/3,"r3":sum(hits)/len(g),"hit1":hits[0],"mrr":rr,"ndcg3":dcg}
rows=[]
for qid,q,g in TESTS:
 b=[x["chunk_id"] for x in baseline[qid]]; f=finals[qid]; bm,fm=m(b,g),m(f,g); rows.append({"qid":qid,"query":q,"gold":next(iter(g)),"baseline_ids":b,"final_ids":f,**{f"base_{k}":v for k,v in bm.items()},**{f"final_{k}":v for k,v in fm.items()},**{f"delta_{k}":fm[k]-bm[k] for k in bm}})
results=pd.DataFrame(rows); display(results)
summary=pd.DataFrame({"baseline":[results[f"base_{k}"].mean() for k in ["p3","r3","hit1","mrr","ndcg3"]],"after":[results[f"final_{k}"].mean() for k in ["p3","r3","hit1","mrr","ndcg3"]]},index=["p3","r3","hit1","mrr","ndcg3"]); summary["delta"]=summary.after-summary.baseline; display(summary.round(3))


In [ ]:
reg=results[results.delta_mrr<0].sort_values("delta_mrr")
if len(reg):
 w=reg.iloc[0]; qid=w.qid; print("Worst regression:",qid); print("Original:",w.query); print("Rewrites:",transformed[qid]); print("Gold:",w.gold,by_id[w.gold]["text"]); print("Baseline:",w.baseline_ids); print("Final:",w.final_ids); print("Inference: the rewrite likely over-emphasized a related concept, pulling a less-direct chunk upward; this is an inference from the ranking change, not proof of model internals.")
else: print("No MRR regression occurred in this ten-query set; that does not prove the transformation is universally safe.")
b,a=summary.baseline,summary.after; display(Markdown(f"### Measured interpretation\nBaseline P@3 **{b.p3:.3f}**, R@3 **{b.r3:.3f}**, Hit@1 **{b.hit1:.3f}**, MRR **{b.mrr:.3f}**, nDCG@3 **{b.ndcg3:.3f}**. After transformation + reranking: **{a.p3:.3f}**, **{a.r3:.3f}**, **{a.hit1:.3f}**, **{a.mrr:.3f}**, **{a.ndcg3:.3f}**. MRR delta: **{summary.loc['mrr','delta']:+.3f}**. Tradeoff: two uncached Gemini calls/query plus extra retrieval and CPU reranking; caching removes repeated Gemini calls."))


## AI use

ChatGPT scaffolded and debugged the notebook and repository documentation. Gemini Flash-Lite performs the query transformations. The saved measurements and final interpretation must come from executing this notebook; do not report unexecuted results.
